# 03 — nn.Module Deep Dive: Layers, Params, Buffers, Init, Hooks, Debugging

Goal: master model engineering in PyTorch.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Parameters vs buffers

- parameters: trainable tensors returned by `.parameters()`
- buffers: non-trainable state saved in state_dict (e.g., running stats)

In [ ]:

import torch, torch.nn as nn

class Mod(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(3, 4)
        self.register_buffer("scale", torch.tensor(0.5))
    def forward(self, x):
        return self.lin(x) * self.scale

m = Mod().to(device)
print("state_dict keys:", list(m.state_dict().keys()))
print("num params:", sum(p.numel() for p in m.parameters()))

## 2. Weight initialization

`torch.nn.init` supports:
- Xavier/Glorot (tanh-ish)
- Kaiming/He (ReLU-ish)

In [ ]:

import torch.nn.init as init
lin = nn.Linear(128, 128).to(device)
init.kaiming_normal_(lin.weight, nonlinearity="relu")
init.zeros_(lin.bias)
print(lin.weight.mean().item(), lin.weight.std().item())

## 3. Normalization layers

- BatchNorm: uses batch stats (train) and running stats (eval)
- LayerNorm: stable for variable batch sizes (Transformers)
- GroupNorm: good for small batches in vision

In [ ]:

bn = nn.BatchNorm1d(16).to(device)
ln = nn.LayerNorm(16).to(device)
gn = nn.GroupNorm(4, 16).to(device)

x = torch.randn(8, 16, device=device)
print(bn(x).shape, ln(x).shape, gn(x).shape)

## 4. Dropout

Active only in `train()` mode.

In [ ]:

drop = nn.Dropout(p=0.5).to(device)
x = torch.ones(10, device=device)
drop.train(); y1 = drop(x)
drop.eval();  y2 = drop(x)
print("train mean:", y1.mean().item(), "eval mean:", y2.mean().item())

## 5. Hooks: capture activations and gradients

Forward hooks:
- activation statistics
Backward hooks:
- gradient statistics

In [ ]:

import torch.nn.functional as F
act = {}
def save_act(name):
    def hook(mod, inp, out):
        with torch.no_grad():
            act[name] = {"mean": out.mean().item(), "std": out.std().item()}
    return hook

mlp = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2)).to(device)
h = mlp[0].register_forward_hook(save_act("layer0"))
_ = mlp(torch.randn(64,10, device=device))
h.remove()
act

## 6. Debugging NaNs/Infs

- `set_detect_anomaly(True)` for locating the op
- monitor activation and gradient norms
- `torch.isfinite`, `nan_to_num`

In [ ]:

torch.autograd.set_detect_anomaly(False)

x = torch.tensor([1.0, 0.0], device=device, requires_grad=True)
y = torch.log(x)  # log(0) -> -inf
loss = y.sum()
loss.backward()
print("y:", y)
print("grad:", x.grad)
print("isfinite:", torch.isfinite(y))